# Gans Scooter – City Data Engineering Pipeline

**Interactive ETL notebook | Wikipedia → Python → MySQL | OpenWeather → Python → MySQL | AeroDataBox/RapidAPI → Python → MySQL**

This portfolio project integrates heterogeneous external data for **Berlin, Hamburg, Munich, Frankfurt and Stuttgart**.

### Pipeline

```text
Wikipedia ───────┐
OpenWeather ─────┼──> Python ETL ──> MySQL ──> Validation
AeroDataBox ─────┘

Extract → Transform → Load → Validate
```

The standalone `gans_scooter_data_pipeline.py` is the reusable ETL implementation. This notebook provides an interactive, documented version for demonstration and analysis.

## 1. Project structure

```text
gans_scooter_data_pipeline/
├── README.md
├── gans_scooter_data_pipeline.py
├── gans_scooter_database.sql
├── gans_scooter_data_pipeline.ipynb
├── requirements.txt
├── .env.example
└── .gitignore
```

### Database model

```text
cities
  ├── populations
  ├── weather
  └── airports
          └── flights
```

## 2. Install dependencies

In [ ]:
# Uncomment if packages are not installed
# %pip install requests beautifulsoup4 mysql-connector-python python-dotenv pandas matplotlib


## 3. Configuration

Create a local `.env` file:

```text
OPENWEATHER_API_KEY=your_openweather_api_key
RAPIDAPI_KEY=your_rapidapi_key
MYSQL_HOST=localhost
MYSQL_PORT=3306
MYSQL_USER=root
MYSQL_PASSWORD=your_mysql_password
MYSQL_DATABASE=gans_cities
```

**Never commit `.env` or API keys to GitHub.**

In [ ]:
import os
import re
from datetime import date, datetime, timedelta

import mysql.connector
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()

OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")

MYSQL_CONFIG = {
    "host": os.getenv("MYSQL_HOST", "localhost"),
    "port": int(os.getenv("MYSQL_PORT", "3306")),
    "user": os.getenv("MYSQL_USER", "root"),
    "password": os.getenv("MYSQL_PASSWORD", ""),
    "database": os.getenv("MYSQL_DATABASE", "gans_cities"),
}

CITIES = {
    "Berlin": {"wikipedia": "https://en.wikipedia.org/wiki/Berlin", "icao": "EDDB"},
    "Hamburg": {"wikipedia": "https://en.wikipedia.org/wiki/Hamburg", "icao": "EDDH"},
    "Munich": {"wikipedia": "https://en.wikipedia.org/wiki/Munich", "icao": "EDDM"},
    "Frankfurt": {"wikipedia": "https://en.wikipedia.org/wiki/Frankfurt", "icao": "EDDF"},
    "Stuttgart": {"wikipedia": "https://en.wikipedia.org/wiki/Stuttgart", "icao": "EDDS"},
}

HTTP_TIMEOUT = 30
RAPIDAPI_HOST = "aerodatabox.p.rapidapi.com"

WIKIPEDIA_HEADERS = {
    "User-Agent": "GansScooterDataPipeline/2.0 (educational portfolio project)"
}

RAPIDAPI_HEADERS = {
    "X-RapidAPI-Key": RAPIDAPI_KEY or "",
    "X-RapidAPI-Host": RAPIDAPI_HOST,
}

print("Cities:", ", ".join(CITIES))


## 4. Validate configuration

In [ ]:
def validate_configuration():
    required = {
        "OPENWEATHER_API_KEY": OPENWEATHER_API_KEY,
        "RAPIDAPI_KEY": RAPIDAPI_KEY,
        "MYSQL_USER": MYSQL_CONFIG["user"],
        "MYSQL_DATABASE": MYSQL_CONFIG["database"],
    }
    missing = [k for k, v in required.items() if not v]
    if missing:
        raise RuntimeError("Missing configuration values: " + ", ".join(missing))
    print("Configuration validation passed.")


def get_mysql_connection():
    return mysql.connector.connect(**MYSQL_CONFIG)


validate_configuration()


## 5. Extract city and population data

Wikipedia is used as the demographic/geographic source. The transformation extracts city name, country, coordinates, population and collection date.

In [ ]:
def parse_coordinate(value):
    if not value:
        return None
    match = re.search(r"-?\d+(?:\.\d+)?", value)
    if not match:
        return None
    coordinate = float(match.group(0))
    if "S" in value.upper() or "W" in value.upper():
        coordinate = -abs(coordinate)
    return coordinate


def extract_population(infobox):
    for row in infobox.find_all("tr"):
        header = row.find("th")
        value = row.find("td")
        if not header or not value:
            continue
        if header.get_text(" ", strip=True).lower() != "population":
            continue

        candidates = re.findall(r"\d[\d,.\s]*", value.get_text(" ", strip=True))
        for raw in candidates:
            cleaned = raw.replace(",", "").replace(".", "").replace(" ", "")
            if cleaned.isdigit() and len(cleaned) >= 4:
                return int(cleaned)
    return None


def scrape_city(city_name, wikipedia_url):
    response = requests.get(wikipedia_url, headers=WIKIPEDIA_HEADERS, timeout=HTTP_TIMEOUT)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    infobox = soup.find("table", class_="infobox")
    if not infobox:
        raise ValueError(f"Wikipedia infobox not found for {city_name}")

    lat_el = soup.find(class_="latitude")
    lon_el = soup.find(class_="longitude")
    latitude = parse_coordinate(lat_el.get_text(strip=True)) if lat_el else None
    longitude = parse_coordinate(lon_el.get_text(strip=True)) if lon_el else None
    population = extract_population(infobox)

    if population is None:
        raise ValueError(f"Population not found for {city_name}")

    return {
        "city": city_name,
        "country": "Germany",
        "latitude": latitude,
        "longitude": longitude,
        "population": population,
        "date_gathered": date.today(),
    }


In [ ]:
city_records = []

for city_name, config in CITIES.items():
    try:
        city_records.append(scrape_city(city_name, config["wikipedia"]))
    except Exception as exc:
        print(f"[ERROR] Wikipedia {city_name}: {exc}")

city_df = pd.DataFrame(city_records)
display(city_df)


## 6. Load cities and population snapshots into MySQL

In [ ]:
def load_city_data(records):
    connection = get_mysql_connection()
    cursor = connection.cursor()

    try:
        for record in records:
            cursor.execute(
                "INSERT INTO cities (city, country, latitude, longitude) "
                "VALUES (%s, %s, %s, %s) "
                "ON DUPLICATE KEY UPDATE country=VALUES(country), "
                "latitude=VALUES(latitude), longitude=VALUES(longitude)",
                (record["city"], record["country"], record["latitude"], record["longitude"]),
            )

            cursor.execute("SELECT city_id FROM cities WHERE city=%s", (record["city"],))
            city_id = cursor.fetchone()[0]

            cursor.execute(
                "INSERT INTO populations (city_id, population, date_gathered) "
                "VALUES (%s, %s, %s) "
                "ON DUPLICATE KEY UPDATE population=VALUES(population)",
                (city_id, record["population"], record["date_gathered"]),
            )

        connection.commit()
        print(f"Loaded {len(records)} city records.")

    except Exception:
        connection.rollback()
        raise
    finally:
        cursor.close()
        connection.close()


load_city_data(city_records)


## 7. Extract current weather

Weather is requested by latitude/longitude using the OpenWeather current-weather endpoint.

Stored fields include temperature, feels-like temperature, rain, wind, snow, sunrise, sunset, description and observation time.

In [ ]:
def get_weather_data(city_name, latitude, longitude):
    response = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={
            "lat": latitude,
            "lon": longitude,
            "appid": OPENWEATHER_API_KEY,
            "units": "metric",
        },
        timeout=HTTP_TIMEOUT,
    )
    response.raise_for_status()

    data = response.json()
    weather = data.get("weather", [])

    return {
        "city": city_name,
        "observation_datetime": datetime.fromtimestamp(data["dt"]),
        "temp": data["main"].get("temp"),
        "feels_like": data["main"].get("feels_like"),
        "rain": data.get("rain", {}).get("1h", 0),
        "wind": data.get("wind", {}).get("speed", 0),
        "snow": data.get("snow", {}).get("1h", 0),
        "sunrise": datetime.fromtimestamp(data["sys"]["sunrise"]),
        "sunset": datetime.fromtimestamp(data["sys"]["sunset"]),
        "weather_description": weather[0].get("description") if weather else None,
    }


In [ ]:
connection = get_mysql_connection()
cursor = connection.cursor(dictionary=True)

try:
    placeholders = ", ".join(["%s"] * len(CITIES))
    cursor.execute(
        f"SELECT city_id, city, latitude, longitude FROM cities WHERE city IN ({placeholders})",
        tuple(CITIES.keys()),
    )
    db_cities = cursor.fetchall()
finally:
    cursor.close()
    connection.close()

weather_records = []

for city in db_cities:
    if city["latitude"] is None or city["longitude"] is None:
        print(f"[WARNING] Missing coordinates: {city['city']}")
        continue
    try:
        weather_records.append(
            get_weather_data(city["city"], float(city["latitude"]), float(city["longitude"]))
        )
    except Exception as exc:
        print(f"[ERROR] Weather {city['city']}: {exc}")

weather_df = pd.DataFrame(weather_records)
display(weather_df)


In [ ]:
def load_weather_data(records):
    connection = get_mysql_connection()
    cursor = connection.cursor(dictionary=True)

    try:
        cursor.execute("SELECT city_id, city FROM cities")
        city_map = {row["city"]: row["city_id"] for row in cursor.fetchall()}

        sql = (
            "INSERT INTO weather (observation_datetime, temp, feels_like, rain, wind, snow, "
            "city_id, sunrise, sunset, weather_description) "
            "VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s) "
            "ON DUPLICATE KEY UPDATE temp=VALUES(temp), feels_like=VALUES(feels_like), "
            "rain=VALUES(rain), wind=VALUES(wind), snow=VALUES(snow), "
            "sunrise=VALUES(sunrise), sunset=VALUES(sunset), "
            "weather_description=VALUES(weather_description)"
        )

        for record in records:
            cursor.execute(
                sql,
                (
                    record["observation_datetime"], record["temp"], record["feels_like"],
                    record["rain"], record["wind"], record["snow"],
                    city_map[record["city"]], record["sunrise"], record["sunset"],
                    record["weather_description"],
                ),
            )

        connection.commit()
        print(f"Processed {len(records)} weather records.")

    except Exception:
        connection.rollback()
        raise
    finally:
        cursor.close()
        connection.close()


load_weather_data(weather_records)


## 8. Airport reference data

| City | ICAO |
|---|---|
| Berlin | EDDB |
| Hamburg | EDDH |
| Munich | EDDM |
| Frankfurt | EDDF |
| Stuttgart | EDDS |

In [ ]:
def load_airports():
    connection = get_mysql_connection()
    cursor = connection.cursor()

    try:
        for city_name, config in CITIES.items():
            cursor.execute("SELECT city_id FROM cities WHERE city=%s", (city_name,))
            result = cursor.fetchone()
            if not result:
                continue

            cursor.execute(
                "INSERT INTO airports (city_id, icao) VALUES (%s,%s) "
                "ON DUPLICATE KEY UPDATE city_id=VALUES(city_id)",
                (result[0], config["icao"]),
            )

        connection.commit()
        print("Airport mappings loaded.")

    except Exception:
        connection.rollback()
        raise
    finally:
        cursor.close()
        connection.close()


load_airports()


## 9. Extract arriving flights

AeroDataBox is accessed through RapidAPI. The notebook retrieves **tomorrow's arrivals** for each airport using two half-day windows.

The stored fields are arrival/departure ICAO, departure airport name, flight number, scheduled arrival and retrieval timestamp.

In [ ]:
def get_flight_data(airport_icao, target_date):
    flights = []

    for start_time, end_time in [("00:00", "11:59"), ("12:00", "23:59")]:
        url = (
            "https://aerodatabox.p.rapidapi.com/"
            f"flights/airports/icao/{airport_icao}/"
            f"{target_date}T{start_time}/{target_date}T{end_time}"
        )

        params = {
            "withLeg": "true",
            "direction": "Arrival",
            "withCancelled": "false",
            "withCodeshared": "true",
            "withCargo": "false",
            "withPrivate": "false",
            "withLocation": "false",
        }

        response = requests.get(
            url, headers=RAPIDAPI_HEADERS, params=params, timeout=HTTP_TIMEOUT
        )
        response.raise_for_status()

        for flight in response.json().get("arrivals", []):
            departure = flight.get("departure", {})
            departure_airport = departure.get("airport", {})
            scheduled = flight.get("arrival", {}).get("scheduledTime", {})

            if not scheduled.get("local"):
                continue

            flights.append({
                "arrival_airport_icao": airport_icao,
                "departure_airport_icao": departure_airport.get("icao"),
                "departure_airport_name": departure_airport.get("name"),
                "flight_number": flight.get("number"),
                "scheduled_arrival_time": scheduled["local"],
                "data_retrieved_at": datetime.now(),
            })

    return flights


target_date = date.today() + timedelta(days=1)
flight_records = []

for city_name, config in CITIES.items():
    try:
        flight_records.extend(get_flight_data(config["icao"], target_date))
    except Exception as exc:
        print(f"[ERROR] Flights {city_name}: {exc}")

flight_df = pd.DataFrame(flight_records)

print("Target date:", target_date)
print("Flight records:", len(flight_df))
display(flight_df.head(10))


In [ ]:
def load_flight_data(records):
    if not records:
        print("No flight records to load.")
        return

    connection = get_mysql_connection()
    cursor = connection.cursor()

    try:
        sql = (
            "INSERT INTO flights (arrival_airport_icao, departure_airport_icao, "
            "departure_airport_name, flight_number, scheduled_arrival_time, data_retrieved_at) "
            "VALUES (%s,%s,%s,%s,%s,%s) "
            "ON DUPLICATE KEY UPDATE data_retrieved_at=VALUES(data_retrieved_at), "
            "departure_airport_name=VALUES(departure_airport_name)"
        )

        for record in records:
            cursor.execute(
                sql,
                (
                    record["arrival_airport_icao"], record["departure_airport_icao"],
                    record["departure_airport_name"], record["flight_number"],
                    record["scheduled_arrival_time"], record["data_retrieved_at"],
                ),
            )

        connection.commit()
        print(f"Processed {len(records)} flight records.")

    except Exception:
        connection.rollback()
        raise
    finally:
        cursor.close()
        connection.close()


load_flight_data(flight_records)


## 10. Validate the database

In [ ]:
def query_df(sql):
    connection = get_mysql_connection()
    try:
        return pd.read_sql(sql, connection)
    finally:
        connection.close()


row_counts = query_df(
    "SELECT 'cities' AS table_name, COUNT(*) AS row_count FROM cities "
    "UNION ALL SELECT 'populations', COUNT(*) FROM populations "
    "UNION ALL SELECT 'weather', COUNT(*) FROM weather "
    "UNION ALL SELECT 'airports', COUNT(*) FROM airports "
    "UNION ALL SELECT 'flights', COUNT(*) FROM flights"
)

display(row_counts)


In [ ]:
latest_weather = query_df(
    "SELECT c.city, w.observation_datetime, w.temp, w.feels_like, "
    "w.rain, w.wind, w.snow, w.weather_description "
    "FROM weather w JOIN cities c ON c.city_id=w.city_id "
    "JOIN (SELECT city_id, MAX(observation_datetime) AS max_dt "
    "FROM weather GROUP BY city_id) latest "
    "ON latest.city_id=w.city_id AND latest.max_dt=w.observation_datetime "
    "ORDER BY c.city"
)

display(latest_weather)


In [ ]:
airport_summary = query_df(
    "SELECT a.icao, c.city, COUNT(f.flight_id) AS flight_count "
    "FROM airports a JOIN cities c ON c.city_id=a.city_id "
    "LEFT JOIN flights f ON f.arrival_airport_icao=a.icao "
    "GROUP BY a.icao, c.city ORDER BY flight_count DESC"
)

display(airport_summary)


In [ ]:
quality_checks = {
    "missing_coordinates": query_df(
        "SELECT city FROM cities WHERE latitude IS NULL OR longitude IS NULL"
    ),
    "invalid_population": query_df(
        "SELECT * FROM populations WHERE population <= 0"
    ),
    "missing_flight_numbers": query_df(
        "SELECT COUNT(*) AS missing_flight_numbers FROM flights "
        "WHERE flight_number IS NULL OR TRIM(flight_number)=''"
    ),
}

for name, result in quality_checks.items():
    print(f"--- {name} ---")
    display(result)


## 11. Example analysis

The resulting data can immediately support simple analytical questions, while the main purpose of the project remains data engineering.

In [ ]:
if not latest_weather.empty:
    display(
        latest_weather[
            ["city", "temp", "feels_like", "weather_description"]
        ].sort_values("temp", ascending=False)
    )


## 12. Engineering practices demonstrated

- Web scraping with `Requests` and `BeautifulSoup`
- REST API integration
- Data transformation and normalization
- Parameterized SQL
- MySQL relational modelling
- Primary and foreign keys
- Unique constraints and indexes
- Upsert/idempotent loading patterns
- Transaction handling
- Environment-variable configuration
- HTTP timeouts and error handling
- Data-quality validation

### Production extensions

Daily scheduling, structured logging, retry/backoff, historical snapshots, automated data-quality tests, Docker, Airflow, CI/CD and cloud deployment.

## 13. Final summary

This project demonstrates a complete **Extract → Transform → Load → Validate** workflow.

Heterogeneous external sources are transformed into structured relational data and stored in MySQL. The resulting model separates city master data, population snapshots, weather observations, airports and flights, providing a strong foundation for analytics and dashboard development.

**Related files:** `README.md`, `gans_scooter_data_pipeline.py`, `gans_scooter_database.sql`, `requirements.txt`, `.env.example`.